# 06 — ML Extensions

Този notebook показва ML extensions: feature matrix, PCA visualization и clustering върху draw pattern features. При лотарийни данни ML моделите трябва да се интерпретират внимателно.


In [ ]:
from pathlib import Path
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"


def read_csv_safe(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def number_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in ["n1", "n2", "n3", "n4", "n5", "n6"] if col in df.columns]


def parse_numbers_text(value) -> list[int]:
    if pd.isna(value):
        return []
    parts = str(value).replace(";", ",").replace("|", ",").split(",")
    nums = []
    for part in parts:
        part = part.strip()
        if part.isdigit():
            nums.append(int(part))
    return nums


def extract_ticket_numbers(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    if len(number_columns(df)) == 6:
        return df.copy()
    text_col = next((c for c in df.columns if "numbers" in c.lower() or "числа" in c.lower()), None)
    if text_col is None:
        return df.copy()
    out = df.copy()
    parsed = out[text_col].apply(parse_numbers_text)
    for i in range(6):
        out[f"n{i+1}"] = parsed.apply(lambda nums: nums[i] if len(nums) > i else np.nan)
    return out


def show_latest_draw(df: pd.DataFrame) -> None:
    if df.empty:
        print("No draw data loaded.")
        return
    latest = df.tail(1).iloc[0]
    nums = [int(latest[col]) for col in number_columns(df)]
    print("Rows:", len(df))
    print("Latest date:", latest.get("date", "n/a"))
    print("Draw number:", latest.get("draw_number", latest.get("draw_no", "n/a")))
    print("Numbers:", nums)


def file_status(paths):
    rows = []
    for path in paths:
        path = Path(path)
        rows.append({
            "file": str(path.relative_to(PROJECT_ROOT)) if str(path).startswith(str(PROJECT_ROOT)) else str(path),
            "exists": path.exists(),
            "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else 0,
        })
    return pd.DataFrame(rows)


draws = read_csv_safe(DATA_DIR / "historical_draws.csv")
normalized = read_csv_safe(DATA_DIR / "v40_normalized_draw_events.csv")
canonical = read_csv_safe(DATA_DIR / "v41_canonical_draw_events.csv")
show_latest_draw(draws)


## Feature matrix


In [ ]:
num_cols = number_columns(draws)
features = pd.DataFrame({
    "sum": draws[num_cols].sum(axis=1),
    "mean": draws[num_cols].mean(axis=1),
    "std": draws[num_cols].std(axis=1),
    "min": draws[num_cols].min(axis=1),
    "max": draws[num_cols].max(axis=1),
    "spread": draws[num_cols].max(axis=1) - draws[num_cols].min(axis=1),
    "odd_count": draws[num_cols].apply(lambda row: sum(int(x) % 2 == 1 for x in row), axis=1),
    "low_count": draws[num_cols].apply(lambda row: sum(int(x) <= 24 for x in row), axis=1),
})
features.tail()


## PCA visualization


In [ ]:
try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    scaled = StandardScaler().fit_transform(features)
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(scaled)
    pca_df = pd.DataFrame(coords, columns=["pc1", "pc2"])
    ax = pca_df.plot(kind="scatter", x="pc1", y="pc2", figsize=(8, 6), title="PCA view of draw pattern features")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    plt.show()
    print("Explained variance ratio:", pca.explained_variance_ratio_)
except Exception as exc:
    print("PCA section skipped:", exc)


## Clustering example


In [ ]:
try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans
    scaled = StandardScaler().fit_transform(features)
    model = KMeans(n_clusters=4, random_state=42, n_init="auto")
    clusters = model.fit_predict(scaled)
    cluster_summary = features.copy()
    cluster_summary["cluster"] = clusters
    display(cluster_summary.groupby("cluster").mean().round(2))
    ax = cluster_summary["cluster"].value_counts().sort_index().plot(kind="bar", figsize=(8, 4), title="Cluster distribution")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Draw count")
    plt.show()
except Exception as exc:
    print("Clustering section skipped:", exc)
